# lazykh video renderer

Run the cells in order to render `exampleVideo/ev`. The final video is downloaded automatically.

To render your own project, upload a `.wav` file and its matching annotated `.txt` script into the same Colab folder, then change `INPUT_FILE` to the shared path without either extension.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPOSITORY_URL = "https://github.com/parubalo/lazykh.git"
BRANCH = "main"
WORKSPACE = Path("/content/lazykh")

if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY_URL, str(WORKSPACE)], check=True)
os.chdir(WORKSPACE)

if shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements.txt"], check=True)

In [ ]:
# Set this to the path shared by your .wav and .txt files, without extensions.
INPUT_FILE = "exampleVideo/ev"
WHISPER_MODEL = "base"

for suffix in (".wav", ".txt"):
    path = Path(INPUT_FILE + suffix)
    if not path.is_file():
        raise FileNotFoundError(f"Missing required input: {path}")

In [ ]:
commands = [
    [sys.executable, "code/gentleScriptWriter.py", "--input_file", INPUT_FILE],
    [sys.executable, "code/whisperAligner.py", "--input_file", INPUT_FILE, "--model", WHISPER_MODEL],
    [sys.executable, "code/scheduler.py", "--input_file", INPUT_FILE],
    [sys.executable, "code/videoDrawer.py", "--input_file", INPUT_FILE, "--use_billboards", "F", "--jiggly_transitions", "F"],
    [sys.executable, "code/videoFinisher.py", "--input_file", INPUT_FILE, "--keep_frames", "F"],
]

for command in commands:
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)

OUTPUT_FILE = Path(INPUT_FILE + "_final.mp4")
if not OUTPUT_FILE.is_file():
    raise FileNotFoundError(f"Rendering did not produce {OUTPUT_FILE}")

In [ ]:
from IPython.display import Video, display

display(Video(filename=str(OUTPUT_FILE), embed=False))

In [ ]:
from google.colab import files

files.download(str(OUTPUT_FILE))